In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.stats import chi2_contingency

In [5]:
expression_data = pd.read_csv('expression_data.csv')

In [26]:
gene_variances = expression_data.var(axis=1)

In [8]:
top_5000_genes = gene_variances.sort_values(ascending=False).head(5000).index

In [9]:
top_variable_expression = expression_data.loc[top_5000_genes]

In [12]:
print(len(top_variable_expression))

5000


In [10]:
print(top_variable_expression.head(5))

       SRR1785238  SRR1785239  SRR1785240  SRR1785241  SRR1785242  SRR1785243  \
28617   29.146046   28.875818   32.691899   32.423348   22.142084   21.530333   
2067   154.923013  152.303056  144.620796  144.620796  174.954663  174.954663   
16178   23.303464   24.785411   32.043544   29.364786   23.099605   23.150761   
19514    8.642984    8.953094   15.021587   12.282399    9.422501    9.833115   
13317  115.124459  112.453515  113.788757  109.734172  108.361017  108.361017   

       SRR1785244  SRR1785245  SRR1785246  SRR1785247  ...  SRR1785316  \
28617    5.966373    5.855159  164.257423  164.257423  ...   45.802692   
2067   117.782800  117.782800  171.500782  171.500782  ...   75.554324   
16178    8.713477    8.909165  136.159832  136.159832  ...   57.007083   
19514    3.984753    3.519763  108.361017  106.999915  ...   23.483012   
13317   61.791317   62.487293  102.294477  102.294477  ...  126.189726   

       SRR1785317  SRR1785318  SRR1785319  SRR1785320  SRR1785321  S

In [14]:
from sklearn.cluster import KMeans

# Assuming `top_variable_expression` contains your subset expression matrix (genes x samples),
# typically samples are rows for clustering, so transpose:
data_for_clustering = top_variable_expression.T  # Samples as rows, genes as features

# Set the number of clusters (k); for gene expression 2-10 is common to try, let's pick 3 here
k = 3
kmeans = KMeans(n_clusters=k, random_state=27)
cluster_labels = kmeans.fit_predict(data_for_clustering)

# cluster_labels is a numpy array with cluster assignment per sample
print(cluster_labels)


[0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 2 2 0 0 1 1 1 1 1 1 1 1 2 2 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 1
 0 0 1 1 1 1 2 2 0 0 0 0 2 2]


In [15]:
k_range = range(2, 11)  # Try k from 2 to 10
cluster_memberships = {}

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=27)
    labels = kmeans.fit_predict(data_for_clustering)
    cluster_memberships[k] = labels
    print(f"Cluster assignments for k={k}:")
    print(labels)  # This is an array of cluster labels per sample
    print()  # Blank line for readability

Cluster assignments for k=2:
[0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 1
 0 0 1 1 1 1 0 0 0 0 0 0 1 1]

Cluster assignments for k=3:
[0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 2 2 0 0 1 1 1 1 1 1 1 1 2 2 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 1
 0 0 1 1 1 1 2 2 0 0 0 0 2 2]

Cluster assignments for k=4:
[0 0 2 0 0 0 1 1 0 0 3 3 0 0 0 0 2 2 0 0 0 0 1 1 0 0 0 0 0 0 0 0 2 2 0 0 0
 0 0 0 0 0 2 2 0 0 1 1 1 1 1 1 1 1 3 3 1 1 1 1 1 1 1 1 1 1 1 1 2 2 0 0 1 1
 0 0 2 2 1 1 3 3 0 0 0 0 2 2]

Cluster assignments for k=5:
[0 0 0 0 0 0 1 1 4 4 3 3 4 4 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 4 4 0 0 4 4 0
 0 4 4 0 0 2 2 1 1 1 1 1 1 1 1 1 1 3 3 1 1 1 1 1 1 1 1 1 1 1 1 0 0 4 4 1 1
 4 4 2 2 1 1 3 3 4 4 1 1 2 2]

Cluster assignments for k=6:
[5 5 0 0 5 5 1 1 4 4 5 5 4 4 5 5 0 0 0 0 0 0 1 1 0 0 5 5 5 5 4 4 0 0 4 4 0
 0 4 4 0 0 2 2 1 1 1 1 1 1 1 1 5 5 3 3 1 1 1 1 1 1 1 1 1

In [30]:
# List of top N values to try
top_n_list = [10, 100, 1000, 10000]

for n in top_n_list:
    # Select top n variable genes
    top_genes = gene_variances.sort_values(ascending=False).head(n).index
    top_variable_expression = expression_data.loc[top_genes]
    data_for_clustering = top_variable_expression.T  # Samples as rows, genes as features

    # Run clustering
    kmeans = KMeans(n_clusters=3, random_state=27)
    cluster_labels = kmeans.fit_predict(data_for_clustering)

    print(f"Cluster labels for top {n} genes:")
    print(cluster_labels)

Cluster labels for top 10 genes:
[0 0 0 0 0 0 1 1 2 2 0 0 0 0 0 0 2 2 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 2 2 0 0 2 2 0 0 0 0 0 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 2 2 0 0 1 1
 2 2 1 1 1 1 1 1 2 2 1 1 1 1]
Cluster labels for top 100 genes:
[0 0 0 0 0 0 1 1 0 0 2 2 0 0 0 0 0 0 0 0 0 0 1 1 0 0 2 2 2 2 0 0 0 0 0 0 0
 0 0 0 0 0 2 2 1 1 1 1 0 0 1 1 0 0 2 2 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 1
 0 0 2 2 1 1 2 2 0 0 1 1 2 2]
Cluster labels for top 1000 genes:
[0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 2 2 0 0 1 1 1 1 1 1 1 1 2 2 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 1
 0 0 1 1 1 1 2 2 0 0 0 0 2 2]
Cluster labels for top 10000 genes:
[0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 2 2 0 0 1 1 1 1 1 1 1 1 2 2 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 1
 0 0 1 1 1 1 2 2 0 0 0 0 2 2]


In [33]:
top_n_list = [10, 100, 1000, 10000]
labels_dict = {}

# Perform clustering for all top-N choices and save labels
for n in top_n_list:
    top_genes = gene_variances.sort_values(ascending=False).head(n).index
    data_for_clustering = expression_data.loc[top_genes].T
    kmeans = KMeans(n_clusters=3, random_state=27)
    labels_dict[n] = kmeans.fit_predict(data_for_clustering)

# Perform chi-squared test on all unique pairs
results = []
for i, n1 in enumerate(top_n_list):
    for n2 in top_n_list[i+1:]:
        contingency = pd.crosstab(labels_dict[n1], labels_dict[n2])
        chi2, p, _, _ = chi2_contingency(contingency)
        results.append({'Genes 1': n1, 'Genes 2': n2, 'Chi2 Statistic': chi2, 'p-value': p})

results_df = pd.DataFrame(results)
print(results_df)

   Genes 1  Genes 2  Chi2 Statistic       p-value
0       10      100       43.193878  9.432314e-09
1       10     1000       55.463571  2.597772e-11
2       10    10000       55.463571  2.597772e-11
3      100     1000       87.108571  5.413817e-18
4      100    10000       87.108571  5.413817e-18
5     1000    10000      176.000000  5.388596e-37


In [ ]:
k = 3
cluster_labels = {}

for n, genes in gene_subsets.items():
    data_subset = expression_data.loc[genes].T  # samples x selected genes
    kmeans = KMeans(n_clusters=k, random_state=27)
    cluster_labels[n] = kmeans.fit_predict(data_subset)
